In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Names of processed dfs:

'''

bnb_chain_processed, bnb_price_processed
arb1_chain_processed, arb2_chain_processed, arb_price_processed 
tron1_chain_processed, tron2_chain_processed, tron_price_processed
avax1_chain_processed, avax2_chain_processed, avax_price_processed
matic1_chain_processed, matic2_chain_processed, matic_price_processed
sol_chain_processed, sol_price_processed
eth1_chain_processed, eth2_chain_processed, eth_price_processed
btc_chain_processed, btc_price_processed

'''

In [ ]:
def clean_data(data, price_data=None, on_chain_data=None, set_index=True):
    if price_data and on_chain_data:
        raise ValueError('Invalid Parameter Values: Both price_data or on_chain Data cannot be True')
    elif price_data:
        prices = data.copy()
        if prices.isna().any().any():
            print('you have nans here')
            return prices 
        
        if set_index:
            prices['time_period_end'] = pd.to_datetime(prices['time_period_end'])
            prices = prices.set_index('time_period_end') 
        else:
            prices.index = pd.to_datetime(prices.index)

        prices['time_open'] = pd.to_datetime(prices['time_open'])
        prices['time_close'] = pd.to_datetime(prices['time_close'])
        return prices
    
    elif on_chain_data:
        metrics = data.copy()
        if metrics.isna().any().any():
            print('you have nans here')
            return metrics 
        
        if set_index:
            metrics['hour'] = pd.to_datetime(metrics['hour'])
            metrics = metrics.set_index('hour')
        else:
            metrics.index = pd.to_datetime(metrics.index)
        return metrics
    else:
        raise ValueError('Invalid Parameter Values: price_data or on_chain Data must be True')
    

In [ ]:
def check_missing_hours(df):
    """
    Check if the DataFrame's datetime index skips any hourly datapoints.
    
    Parameters:
        df (pd.DataFrame): DataFrame with a DatetimeIndex.
        
    Returns:
        missing (pd.DatetimeIndex): The missing hourly timestamps.
    """
    # Ensure the index is a DatetimeIndex
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("DataFrame index must be a DatetimeIndex.")
    
    # Create an expected date_range from the minimum to the maximum timestamp at hourly frequency
    expected_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='H')
    
    # Determine which timestamps are missing
    missing = expected_range.difference(df.index)
    return missing

In [ ]:
def preprocess_data(data, price_data=None, on_chain_data=None, set_index=True):
    if price_data and on_chain_data:
        raise ValueError('Invalid Parameter Values. Both price_data and on_chain_data cannot both be True')

    elif (not price_data) and (not on_chain_data):
        raise ValueError('Invalid Parameter Values. Both price_data and on_chain_data cannot both be False')
    
    else:
        # Create the target df and merge on correct dates. Then forward fill the na values
        if price_data:
            df = clean_data(data.copy(), price_data=True, set_index=True) if set_index else clean_data(data.copy(), price_data=True, set_index=False)
            target_df = pd.DataFrame(0, columns=[0], index=pd.date_range(start=df.index.min(), end=df.index.max(), freq='H'))
        
        elif on_chain_data:
            df = clean_data(data.copy(), on_chain_data=True, set_index=True) if set_index else clean_data(data.copy(), on_chain_data=True, set_index=False)
            target_df = pd.DataFrame(0, columns=[0], index=pd.date_range(start=df.index.min(), end=df.index.max(), freq='H'))

        target_df = target_df.join(df, how='left').drop(0, axis=1)
        target_df = target_df.fillna(method='ffill')


        return target_df

In [ ]:
def round_hours(staking_dat):
    staking_data = staking_dat.copy()
    staking_data['createdAt'] = pd.to_datetime(
    staking_data['createdAt'],
    format='mixed',
    utc=True,
    dayfirst=False)
    staking_data['createdAt_rounded'] = (staking_data['createdAt'] + pd.Timedelta(microseconds=1)).dt.ceil('H')

    return staking_data

BITCOIN CLEANING

In [ ]:
btc = pd.read_parquet('BTC_Hourly_On_Chain_Data_sorted.parquet')
btc.head()

In [ ]:
# We must shift the BTC Data by 1, since we don't know that information until then

btc['hour'] = pd.to_datetime(btc['hour'])
btc = btc.set_index('hour')
btc = btc.shift(1)
btc = btc.dropna()
display(btc.head())
init_num_missing_hours = len(check_missing_hours(btc))
num_nils_btc = np.where(btc == '<nil>', 1, 0).sum()
print(f'Number of Nils in raw btc on-chain Data: {num_nils_btc}')
print(f'Number of missing hours in the raw BTC on-chain Data: {init_num_missing_hours}')

In [ ]:
btc_chain_processed = preprocess_data(data=btc, on_chain_data=True, set_index=False)
display(btc_chain_processed.head())
num_missing_hours = len(check_missing_hours(btc_chain_processed))
print(f'Number of missing hours in the processed BTC on-chain Data: {num_missing_hours}')

In [ ]:
btc_price = pd.read_csv('btc_hourly.csv').sort_values('time_period_end').set_index('time_period_end')
btc_price.index = pd.to_datetime(btc_price.index)
display(btc_price.head())
num_nans_btc_price = btc.isna().sum().sum()
print(f'Number of NaNs in raw btc price Data: {num_nans_btc_price}')
init_num_missing_hours_price = len(check_missing_hours(btc_price))
print(f'Number of missing hours in the raw BTC price Data: {init_num_missing_hours_price}')

In [ ]:
btc_price_processed = preprocess_data(data=btc_price, price_data=True, set_index=False)
display(btc_price_processed.head())
num_missing_hours_price = len(check_missing_hours(btc_price_processed))
print(f'Number of missing hours in the processed BTC price Data: {num_missing_hours_price}')

ETHEREUM CLEANING

In [ ]:
eth = pd.read_parquet('ETH_Hourly_On_Chain_Data_sorted.parquet').reset_index().drop('index', axis=1).shift(1).iloc[1:]
eth.head()

In [ ]:
print(np.where(eth['average_total_difficulty'] == '<nil>', 1, 0).sum())
print(np.where(eth['average_total_difficulty'] == '<nil>'))
print(np.where(eth['average_base_fee_per_gas'] == '<nil>', 1, 0).sum())
print(np.where(eth['average_base_fee_per_gas'] == '<nil>'))
print(np.where(eth['average_blob_gas_used'] == '<nil>', 1, 0).sum())
print(np.where(eth['average_excess_blob_gas'] == '<nil>', 1, 0).sum())

In [ ]:
# Cutoff ETH at the end because we have NIL values for difficulty
# Get rid of blob metrics because these are only recorded after ethereum moved to proof of stake

eth_cutoff = eth.set_index('hour').loc[:'2022-09-15 07'][['average_number', 'average_gas_limit', 'average_gas_used', 'average_difficulty', 'average_size']]
# Cutoff so we can include average_base_fee_per_gas in our feature set
eth_cutoff.index = pd.to_datetime(eth_cutoff.index)
eth_cutoff.iloc[52740:].head(10)

In [ ]:
eth1 = eth_cutoff.loc[:'2021-08-05 11']
eth2 = eth_cutoff.loc['2021-08-05 12':]

In [ ]:
num_missing_hours_chain_eth1 = len(check_missing_hours(eth1))
num_missing_hours_chain_eth2 = len(check_missing_hours(eth2))
print(f'Number of missing hours in raw eth1 on-chain Data: {num_missing_hours_chain_eth1}')
print(f'Number of missing hours in raw eth2 on-chain Data: {num_missing_hours_chain_eth2}')

In [ ]:
eth1_chain_processed = preprocess_data(data=eth1, on_chain_data=True, set_index=False)
eth2_chain_processed = preprocess_data(data=eth2, on_chain_data=True, set_index=False)
num_missing_hours_chain_eth1_processed = len(check_missing_hours(eth1_chain_processed))
print(f'Number of missing hours in raw eth2 on-chain Data: {num_missing_hours_chain_eth1_processed}')

In [ ]:
eth_price = pd.read_csv('eth_hourly.csv').set_index('time_period_end')
eth_price.index = pd.to_datetime(eth_price.index)
num_nans = eth_price.isna().sum().sum()
num_missing_hours_eth_price = len(check_missing_hours(eth_price))
print(f'Number of NaNs in raw eth price Data: {num_nans}')
print(f'Number of missing hours in raw eth price Data: {num_missing_hours_eth_price}')
eth_price_processed = preprocess_data(data=eth_price, price_data=True, set_index=False)
eth_price_processed.head()

SOLANA CLEANING

In [ ]:
# Solana On-Chain

raw_sol_chain = pd.read_parquet('query_result_SOL.parquet').set_index('hour').shift(1).iloc[1:]
raw_sol_chain.index = pd.to_datetime(raw_sol_chain.index)
raw_sol_chain

In [ ]:
raw_sol_chain = raw_sol_chain[['average_total_transactions', 'average_successful_transactions', 'average_failed_transactions', 'average_total_vote_transactions', 'average_total_non_vote_transactions', 'average_successful_vote_transactions', 'average_successful_non_vote_transactions', 'average_failed_vote_transactions', 'average_failed_non_vote_transactions']]
num_nils_sol_chain = np.where(raw_sol_chain == '<nil>', 1, 0).sum()
print(f'Number of Nils in raw SOL on-chain Data: {num_nils_sol_chain}')
init_num_missing_hours_sol_chain = len(check_missing_hours(raw_sol_chain))
print(f'Number of missing hours in the raw SOL on-chain Data: {init_num_missing_hours_sol_chain}')

In [ ]:
sol_chain_processed = preprocess_data(data=raw_sol_chain, on_chain_data=True, set_index=False).iloc[1:]
processed_num_missing_hours_sol_chain = len(check_missing_hours(sol_chain_processed))
print(f'Number of missing hours in the processed SOL on-chain Data: {processed_num_missing_hours_sol_chain}')
sol_chain_processed.head()

In [ ]:
# SOL Price Data

raw_sol_price = pd.read_csv('sol_hourly.csv').set_index('time_period_end')
raw_sol_price.index = pd.to_datetime(raw_sol_price.index)
init_num_nans_sol_price = raw_sol_price.isna().sum().sum()
print(f'Number of NaNs in the raw SOL price Data: {init_num_nans_sol_price}')
init_num_missing_hours_sol_price = len(check_missing_hours(raw_sol_price))
print(f'Number of missing hours in the raw SOL price Data: {init_num_missing_hours_sol_price}')
raw_sol_price.head()

In [ ]:
sol_price_processed = preprocess_data(data=raw_sol_price, price_data=True, set_index=False)
processed_num_missing_hours_sol_price = len(check_missing_hours(sol_price_processed))
print(f'Number of missing hours in the processed SOL price Data: {processed_num_missing_hours_sol_price}')
sol_price_processed.head()

MATIC (POLYGON) CLEANING

In [ ]:
# matic on-chain Data

raw_matic_chain = pd.read_parquet('query_result_MATIC.parquet').set_index('hour').shift(1).iloc[1:]
raw_matic_chain.index = pd.to_datetime(raw_matic_chain.index)
display(raw_matic_chain)
total_matic_nils = np.where(raw_matic_chain['average_base_fee_per_gas'] == '<nil>', 1, 0).sum()
print(f'Total number of Nils in Raw MATIC on-chain Data: {total_matic_nils}')
other_nils_matic = np.where(raw_matic_chain[raw_matic_chain.columns[:-1]] == '<nil>', 1, 0).sum()
print(f'Number of Nils other than average_base_fee_per_gas {other_nils_matic}')
num_missing_hours_chain_matic_raw = len(check_missing_hours(raw_matic_chain))
print(f'Number of missing hours for raw MATIC on-chain Data {num_missing_hours_chain_matic_raw}')

In [ ]:
# Split MATIC into matic1 and matic2 then process them
matic1_chain = raw_matic_chain.iloc[:14338][raw_matic_chain.columns[1:-1]]
matic2_chain = raw_matic_chain.iloc[14338:, 1:]

matic1_chain_processed = preprocess_data(data=matic1_chain, on_chain_data=True, set_index=False)
matic2_chain_processed = preprocess_data(data=matic2_chain, on_chain_data=True, set_index=False) #matic2 includes the 'average_base_fee_per_gas' field

num_missing_hours_chain_matic1_processed = len(check_missing_hours(matic1_chain_processed))
num_missing_hours_chain_matic2_processed = len(check_missing_hours(matic2_chain_processed))
print(f'Number of missing hours for processed MATIC1 on-chain Data {num_missing_hours_chain_matic1_processed}')
print(f'Number of missing hours for processed MATIC2 on-chain Data {num_missing_hours_chain_matic2_processed}')

In [ ]:
# matic price Data

raw_matic_price = pd.read_csv('matic_hourly.csv').set_index('time_period_end')
raw_matic_price.index = pd.to_datetime(raw_matic_price.index)
init_num_nans_matic_price = raw_matic_price.isna().sum().sum()
print(f'Number of NaNs in the raw MATIC price Data: {init_num_nans_matic_price}')
init_num_missing_hours_matic_price = len(check_missing_hours(raw_matic_price))
print(f'Number of missing hours in the raw MATIC price Data: {init_num_missing_hours_matic_price}')
matic_price_processed = preprocess_data(data=raw_matic_price, price_data=True, set_index=False)
processed_num_missing_hours_matic_price = len(check_missing_hours(matic_price_processed))
print(f'Number of missing hours in the processed MATIC price Data: {processed_num_missing_hours_matic_price}')
matic_price_processed.head()

AVALANCHE CLEANING

In [ ]:
# AVAX on-chain Data
raw_avax_chain = pd.read_parquet('query_result_AVALANCHE.parquet').set_index('hour').shift(1).iloc[1:]
raw_avax_chain.index = pd.to_datetime(raw_avax_chain.index)
raw_avax_chain = raw_avax_chain[raw_avax_chain.columns[1:]]
display(raw_avax_chain.head())
total_avax_nils = np.where(raw_avax_chain == '<nil>', 1, 0).sum()
print(f'Total number of Nils in raw AVAX on-chain Data: {total_avax_nils}')
num_missing_hours_chain_avax_raw = len(check_missing_hours(raw_avax_chain))
print(f'Number of missing hours for raw AVAX on-chain Data {num_missing_hours_chain_avax_raw}')

In [ ]:
avax1 = raw_avax_chain.iloc[:5593][raw_avax_chain.columns[:-1]]
avax2 = raw_avax_chain.iloc[5593:]
avax1.head()

In [ ]:
# Most of the datapoints stop skipping hours by 2021. But I will forward fill since 2020 and make that discretionary decision later.

avax1_chain_processed = preprocess_data(data=avax1, on_chain_data=True, set_index=False)
avax2_chain_processed = preprocess_data(data=avax2, on_chain_data=True, set_index=False)
total_avax1_nils = np.where(avax1_chain_processed == '<nil>', 1, 0).sum()
total_avax2_nils = np.where(avax2_chain_processed == '<nil>', 1, 0).sum()
print(f'Total number of Nils in processed AVAX1 on-chain Data: {total_avax1_nils}')
print(f'Total number of Nils in processed AVAX2 on-chain Data: {total_avax2_nils}')
num_missing_hours_chain_avax1_pro = len(check_missing_hours(avax1_chain_processed))
print(f'Number of missing hours for processed AVAX1 on-chain Data {num_missing_hours_chain_avax1_pro}')
num_missing_hours_chain_avax2_pro = len(check_missing_hours(avax2_chain_processed))
print(f'Number of missing hours for processed AVAX2 on-chain Data {num_missing_hours_chain_avax2_pro}')

In [ ]:
# AVAX Price Data

raw_avax_price = pd.read_csv('avax_hourly.csv').set_index('time_period_end')
raw_avax_price.index = pd.to_datetime(raw_avax_price.index)
init_num_nans_avax_price = raw_avax_price.isna().sum().sum()
print(f'Number of NaNs in the raw AVAX price Data: {init_num_nans_avax_price}')
init_num_missing_hours_avax_price = len(check_missing_hours(raw_avax_price))
print(f'Number of missing hours in the raw AVAX price Data: {init_num_missing_hours_avax_price}')
avax_price_processed = preprocess_data(data=raw_avax_price, price_data=True, set_index=False)
processed_num_missing_hours_avax_price = len(check_missing_hours(avax_price_processed))
print(f'Number of missing hours in the processed AVAX price Data: {processed_num_missing_hours_avax_price}')
avax_price_processed.head()

TRON CLEANING

In [ ]:
# TRON on-chain Data
raw_tron_chain = pd.read_parquet('TRON_Hourly_On_Chain_Data_sorted.parquet').set_index('hour').shift(1).iloc[1:]
raw_tron_chain.index = pd.to_datetime(raw_tron_chain.index)
raw_tron_chain = raw_tron_chain[raw_tron_chain.columns[1:]]
display(raw_tron_chain)
total_tron_nils = np.where(raw_tron_chain == '<nil>', 1, 0).sum()
print(f'Total number of Nils in raw TRON on-chain Data: {total_tron_nils}')
num_missing_hours_chain_tron_raw = len(check_missing_hours(raw_tron_chain))
print(f'Number of missing hours for raw TRON on-chain Data {num_missing_hours_chain_tron_raw}')

In [ ]:
raw_tron_chain = raw_tron_chain[['average_gas_limit', 'average_gas_used', 'average_size', 'unique_number_of_miners']]
tron_chain_processed = preprocess_data(data=raw_tron_chain, on_chain_data=True, set_index=False)
total_tron_nils = np.where(tron_chain_processed == '<nil>', 1, 0).sum()
print(f'Total number of Nils in processed TRON on-chain Data: {total_tron_nils}')
num_missing_hours_chain_tron_pro = len(check_missing_hours(tron_chain_processed))
print(f'Number of missing hours for processed TRON on-chain Data {num_missing_hours_chain_tron_pro}')
tron_chain_processed.head()

In [ ]:
print(np.where(tron_chain_processed['average_gas_limit'] == 0.0, 1, 0).sum())
print(np.where(tron_chain_processed['average_gas_used'] == 0.0, 1, 0).sum())

In [ ]:
# Split the tron Data
tron1_chain_processed = tron_chain_processed.iloc[:2623][['average_size', 'unique_number_of_miners']]
tron2_chain_processed = tron_chain_processed.iloc[2623:]

In [ ]:
# TRON Price Data

raw_tron_price = pd.read_csv('tron_hourly.csv').set_index('time_period_end')
raw_tron_price.index = pd.to_datetime(raw_tron_price.index)
init_num_nans_tron_price = raw_tron_price.isna().sum().sum()
print(f'Number of NaNs in the raw TRON price Data: {init_num_nans_tron_price}')
init_num_missing_hours_tron_price = len(check_missing_hours(raw_tron_price))
print(f'Number of missing hours in the raw TRON price Data: {init_num_missing_hours_tron_price}')
tron_price_processed = preprocess_data(data=raw_tron_price, price_data=True, set_index=False)
processed_num_missing_hours_tron_price = len(check_missing_hours(tron_price_processed))
print(f'Number of missing hours in the processed TRON price Data: {processed_num_missing_hours_tron_price}')
tron_price_processed.head()

ARBITRUM CLEANING

In [ ]:
# ARBITRUM on-chain Data
raw_arb_chain = pd.read_parquet('ARB_Hourly_On_Chain_Data_sorted.parquet').set_index('hour').shift(1).iloc[1:]
raw_arb_chain.index = pd.to_datetime(raw_arb_chain.index)
raw_arb_chain = raw_arb_chain[raw_arb_chain.columns[1:]]
display(raw_arb_chain)
total_arb_nils = np.where(raw_arb_chain == '<nil>', 1, 0).sum()
print(f'Total number of Nils in raw ARB on-chain Data: {total_arb_nils}')
num_missing_hours_chain_arb_raw = len(check_missing_hours(raw_arb_chain))
print(f'Number of missing hours for raw ARB on-chain Data {num_missing_hours_chain_arb_raw}')

In [ ]:
print(np.where(raw_arb_chain['average_total_difficulty'] == 0.0, 1, 0).sum())
print(np.where(raw_arb_chain['average_nonce'] == 0.0, 1, 0).sum())
print(np.where(raw_arb_chain['average_base_fee_per_gas'] == '<nil>', 1, 0).sum())

In [ ]:
# Split the ARB On-Chain Data
raw_arb1_chain = raw_arb_chain[['average_gas_limit', 'average_gas_used', 'average_size', 'unique_number_of_miners']].iloc[:10945]
raw_arb2_chain = raw_arb_chain.iloc[10945:]
display(raw_arb1_chain.head())
raw_arb2_chain.head()

In [ ]:
# Process the ARB Data

arb1_chain_processed = preprocess_data(data=raw_arb1_chain, on_chain_data=True, set_index=False)
arb2_chain_processed = preprocess_data(data=raw_arb2_chain, on_chain_data=True, set_index=False)
total_arb1_nils = np.where(arb1_chain_processed == '<nil>', 1, 0).sum()
print(f'Total number of Nils in processed ARB1 on-chain Data: {total_arb1_nils}')
total_arb2_nils = np.where(arb2_chain_processed == '<nil>', 1, 0).sum()
print(f'Total number of Nils in processed ARB2 on-chain Data: {total_arb1_nils}')
num_missing_hours_chain_arb1_pro = len(check_missing_hours(arb1_chain_processed))
print(f'Number of missing hours for processed ARB1 on-chain Data {num_missing_hours_chain_arb1_pro}')
num_missing_hours_chain_arb2_pro = len(check_missing_hours(arb2_chain_processed))
print(f'Number of missing hours for processed ARB2 on-chain Data {num_missing_hours_chain_arb2_pro}')
display(arb1_chain_processed.head())
arb2_chain_processed.head()

In [ ]:
# ARBITRUM Price Data

raw_arb_price = pd.read_csv('arbitrum_hourly.csv').set_index('time_period_end')
raw_arb_price.index = pd.to_datetime(raw_arb_price.index)
init_num_nans_arb_price = raw_arb_price.isna().sum().sum()
print(f'Number of NaNs in the raw ARB price Data: {init_num_nans_arb_price}')
init_num_missing_hours_arb_price = len(check_missing_hours(raw_arb_price))
print(f'Number of missing hours in the raw ARB price Data: {init_num_missing_hours_arb_price}')
arb_price_processed = preprocess_data(data=raw_arb_price, price_data=True, set_index=False)
processed_num_missing_hours_arb_price = len(check_missing_hours(arb_price_processed))
print(f'Number of missing hours in the processed ARB price Data: {processed_num_missing_hours_arb_price}')
arb_price_processed.head()

BNB CLEANING

In [ ]:
# BNB on-chain Data
raw_bnb_chain = pd.read_parquet('query_result_BNB.parquet').set_index('hour').shift(1).iloc[1:]
raw_bnb_chain.index = pd.to_datetime(raw_bnb_chain.index)
raw_bnb_chain = raw_bnb_chain[raw_bnb_chain.columns[1:-1]]
display(raw_bnb_chain)
total_bnb_nils = np.where(raw_bnb_chain == '<nil>', 1, 0).sum()
print(f'Total number of Nils in raw BNB on-chain Data: {total_bnb_nils}')
num_missing_hours_chain_bnb_raw = len(check_missing_hours(raw_bnb_chain))
print(f'Number of missing hours for raw BNB on-chain Data {num_missing_hours_chain_bnb_raw}')

In [ ]:
bnb_chain_processed = preprocess_data(data=raw_bnb_chain, on_chain_data=True, set_index=False)
num_missing_hours_chain_bnb_pro = len(check_missing_hours(bnb_chain_processed))
print(f'Number of missing hours for processed BNB on-chain Data {num_missing_hours_chain_bnb_pro}')
bnb_chain_processed.head()

In [ ]:
# BNB Price Data

raw_bnb_price = pd.read_csv('bnb_hourly.csv').set_index('time_period_end')
raw_bnb_price.index = pd.to_datetime(raw_bnb_price.index)
init_num_nans_bnb_price = raw_bnb_price.isna().sum().sum()
print(f'Number of NaNs in the raw BNB price Data: {init_num_nans_bnb_price}')
init_num_missing_hours_bnb_price = len(check_missing_hours(raw_bnb_price))
print(f'Number of missing hours in the raw BNB price Data: {init_num_missing_hours_bnb_price}')
bnb_price_processed = preprocess_data(data=raw_bnb_price, price_data=True, set_index=False)
processed_num_missing_hours_bnb_price = len(check_missing_hours(bnb_price_processed))
print(f'Number of missing hours in the processed BNB price Data: {processed_num_missing_hours_bnb_price}')
bnb_price_processed.head()